# Topic 09 — Practice: Apply / Map / Transform

**The notebook is the lesson.** Work top to bottom. Cells with `assert` grade themselves — a silent run = pass, an `AssertionError` = fix your code. Warm-Up, Interview Lens and Reflection have **no answer key** — answer in writing.

_Spaced repetition: ~60% of this notebook is the current topic, ~40% revisits earlier topics._

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.max_columns', 30)
RAW = '../datasets/raw/'
products = pd.read_csv(RAW+'products.csv', dtype={'product_id':str})
products['list_price'] = np.where(products['list_price']<0, np.nan, products['list_price'])
orders = pd.read_csv(RAW+'orders.csv', dtype={'order_id':str,'customer_id':str})
orders['channel'] = orders['channel'].str.strip().str.title()
orders['status'] = orders['status'].str.strip().str.lower()
orders['order_date'] = pd.to_datetime(orders['order_date'], format='mixed', dayfirst=True, errors='coerce')
orders['month'] = orders['order_date'].dt.to_period('M').astype(str)
items = pd.read_csv(RAW+'order_items.csv', dtype={'order_id':str,'product_id':str})
items['line_revenue'] = items['quantity']*items['unit_price']*(1-items['discount'])
master = (items.merge(products[['product_id','category','unit_cost']], on='product_id', how='left')
              .merge(orders[['order_id','channel','status','month','order_date']], on='order_id', how='left'))
master['line_cogs'] = master['quantity']*master['unit_cost']
master['line_profit'] = master['line_revenue'] - master['line_cogs']
import time
customers = pd.read_csv(RAW+'customers.csv', dtype={'customer_id':str}).drop_duplicates('customer_id')


## 🔁 Warm-Up Recall (earlier topics — no answers given)

From **Topics 01–08**:

1. (T06) What do `validate=` and `indicator=` protect you from?
2. (T08) Why pass `na=False` to `str.contains`?
3. NumPy: what does `np.select` do that `np.where` can't?

In [ ]:
# NumPy recall: multi-branch banding with np.select
q = np.array([3, 12, 150, 8, 99])
bands = ...   # bulk>=100, medium>=10, else small
assert list(bands) == ['small','medium','bulk','small','medium']
print('ok')

## 🎯 Core Lesson Tasks (current topic)

Toolbox fastest→slowest: vectorized/np.where/np.select → map → apply. Avoid apply(axis=1) for math.

**Core 1 — map lookup.** Map lowercased `segment` to `{consumer→B2C, business→B2B, pro athlete→B2C}` into `seg2`.

In [ ]:
seg = {'consumer':'B2C','business':'B2B','pro athlete':'B2C'}
customers['seg2'] = ...
assert customers['seg2'].isin(['B2C','B2B']).any()
print(customers['seg2'].value_counts(dropna=False))

**Core 2 — vectorized bands.** `margin_band` via `np.where` (profit/loss); `size_band` via `np.select` (bulk≥100, medium≥10, else small).

In [ ]:
master['margin_band'] = ...
master['size_band'] = ...
assert set(master['size_band'].dropna().unique()) <= {'bulk','medium','small'}
print(master['size_band'].value_counts())

**Core 3 — prove vectorization wins.** Compute line revenue vectorized vs `apply(axis=1)`; compare time & equality.

In [ ]:
t0=time.time(); v = master['quantity']*master['unit_price']*(1-master['discount']); tv=time.time()-t0
t0=time.time(); a = master.apply(lambda r: r['quantity']*r['unit_price']*(1-r['discount']), axis=1); ta=time.time()-t0
assert np.allclose(v, a, equal_nan=True)
print(f'vectorized {tv:.4f}s  apply {ta:.4f}s  -> {ta/max(tv,1e-9):.0f}x')

## 🔀 Mixed Review Tasks (earlier topics — spaced repetition)

**Mixed review (Topic 07) — late flag.** From `shipments`, flag `is_late` (shipped after promised), vectorized, dtype bool.

In [ ]:
ship = pd.read_csv(RAW+'shipments.csv', dtype={'order_id':str}, parse_dates=['promised_date','shipped_date'])
ship['is_late'] = ...
assert ship['is_late'].dtype == bool
print('late shipments:', ship['is_late'].sum())

**Mixed review (Topic 05/06) — group the features.** Mean `line_profit` by `size_band`.

In [ ]:
profit_by_band = ...
assert len(profit_by_band) <= 3
profit_by_band

## 🕵️ Data Detective Investigation

**Case file #9 — the analyst feature set.** Engineer per-order features the capstone needs: `revenue`, `is_returned`, `days_to_ship`, `is_late`. Use vectorized ops only.

In [ ]:
ship = pd.read_csv(RAW+'shipments.csv', dtype={'order_id':str}, parse_dates=['promised_date','shipped_date'])
ship['days_to_ship'] = (ship['shipped_date'] - ship['promised_date']).dt.days
ship['is_late'] = ship['shipped_date'] > ship['promised_date']
order_rev = master.groupby('order_id')['line_revenue'].sum().rename('revenue')
features = order_rev.to_frame().join(ship.set_index('order_id')[['days_to_ship','is_late']])
features['is_returned'] = orders.set_index('order_id')['status'].eq('returned')
assert features['is_late'].dropna().dtype == bool
print(features.head())

## 🔎 Interview Lens (write answers — no model answers)

- **Q26:** Why is vectorization faster than `iterrows`/loops? What's happening underneath?
- **Q27:** When is `apply` a code smell, and what replaces it?
- **Q28:** How do `category` dtypes save memory and speed up groupby? When can they hurt?

## ✍️ Reflection (write short explanations)

1. Answer Q26 and Q27 in writing.
2. What was your apply-vs-vectorized speed ratio?
3. **Investigation log:** which engineered features matter most for the capstone?